# Acervo que Fala — Notebook 04 (v3): fechando dois bugs de prompt

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

Este notebook fecha o pipeline: além do alt-text, entram a **descrição do objeto (nível 2)** e as **flags de divergência** — rodando num lote de **20 objetos** (os 5 do smoke test + 15 novos, sorteados fora do conjunto de avaliação).

**O que mudou da v2 para a v3:** a v2 aplicou as 12 regras da revisão editorial e acertou quase tudo — zero menções à fotografia no nível 2, medidas viraram escala, aves das penas citadas. Mas a análise dos resultados (achado por checagem de código, não pela revisão visual do Eduardo) encontrou **2 bugs no prompt v5**:

1. A regra "artefato nunca aparece" só estava escrita para o alt-text — a Flauta descreveu a numeração no nível 2, mesmo com a flag correta gerada.
2. A atribuição ao registro do museu ("segundo o registro do museu...") sumiu em 15 dos 20 textos — o prompt só garantia isso via um exemplo específico, e o modelo não generalizou.

O **prompt v6** fecha os dois: a regra do artefato agora vale para as duas saídas, com exemplo errado×certo em cada uma; e a atribuição virou exigência explícita (uma marca obrigatória antes do primeiro fato do catálogo, com fórmulas alternativas para não soar repetitivo).

*Metodologia: projeto construído por um designer com LLMs como suporte (vibe coding) — cada célula explicada.*

### Como rodar
1. **Ambiente de execução → Alterar o tipo → GPU T4** · 2. **Executar tudo** · 3. Tempo: **~35–45 min** (20 objetos × 2 gerações). Deixe a aba aberta durante a execução.

In [ ]:
# Etapa 1 — Instalação (Pillow travada, regra da casa) + checagem do ambiente
import PIL
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers pillow=={PIL.__version__}
import torch, transformers
from PIL import ImageDraw
from torchvision.io import decode_image
print(f"transformers {transformers.__version__} | GPU: {torch.cuda.is_available()}")
print("ambiente íntegro ✓")

## Etapa 2 — Buscar os 20 objetos, com salvaguardas de imagem (~3 min)

O lote: os **5 objetos do smoke test** (para comparar com os notebooks anteriores) + **15 novos**, sorteados com seed fixa (reproduzível) entre os itens que **não** estão no conjunto de avaliação — o lote serve para testar o pipeline em escala, sem "viciar" nos itens que depois vão dar a nota.

Duas salvaguardas novas ao baixar cada foto:

- **Orientação EXIF**: fotos de câmera guardam a rotação numa etiqueta interna que os navegadores aplicam, mas o Python não — sem esta linha, o modelo poderia receber uma foto deitada sem ninguém saber. `ImageOps.exif_transpose` aplica a rotação correta.
- **Conversão para RGB**: garante que qualquer foto (escala de cinza, outros formatos de cor) chegue ao modelo no formato esperado.

Desta vez o registro completo de cada objeto (povo, materiais, dimensões, descrição curatorial...) **viaja junto** — a correção do bug do Notebook 03.

In [ ]:
import io, re, requests
from PIL import Image, ImageOps

BASE = "https://tainacan.museudoindio.gov.br/wp-json/tainacan/v2"
IDS_SMOKE = [9196, 665, 51023, 63283, 78838]
# 15 novos: sorteio seed 42, estratificado por categoria, excluindo os 50 casos
# de avaliação (seleção documentada no repositório, commit da E7)
IDS_LOTE = [1376, 84811, 883523, 2081, 5011, 200648, 210680, 5146, 500179, 3411, 1366, 4156, 205095, 905, 500322]

CAMPOS_REGISTRO = ["Nome do item", "Povo", "Categoria", "Matéria-prima",
                   "Técnica de confecção", "Dimensões", "Função",
                   "Estado de origem", "Ano de aquisição do objeto", "Descrição"]

objetos = []
for item_id in IDS_SMOKE + IDS_LOTE:
    item = requests.get(f"{BASE}/items/{item_id}", timeout=60).json()
    url_imagem = re.search(r'src="([^"]+)"', item["document_as_html"]).group(1)
    foto = Image.open(io.BytesIO(requests.get(url_imagem, timeout=90).content))
    foto = ImageOps.exif_transpose(foto).convert("RGB")  # salvaguardas
    meta_bruto = requests.get(f"{BASE}/item/{item_id}/metadata", timeout=60).json()
    todos = {m["metadatum"]["name"]: m["value_as_string"] for m in meta_bruto if m.get("value_as_string")}
    registro = {c: todos.get(c, "") for c in CAMPOS_REGISTRO}
    objetos.append({"id": item_id, "titulo": item["title"], "foto": foto, "registro": registro})
    print(f"✓ {item_id} — {item['title']} ({registro['Povo']})")
print(f"{len(objetos)} objetos carregados")

In [ ]:
# Etapa 3 — Drive (rubrica v1.1) + embeddings do RAG (como no Notebook 03)
import json, os
from google.colab import drive
from sentence_transformers import SentenceTransformer, util

drive.mount("/content/drive")
PROJETO = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM"
# v1.1: rubrica atualizada com as regras da revisão editorial (arquivo novo,
# versionado — a v1.0 continua no Drive como rubrica.json)
with open(f"{PROJETO}/dados/rubrica_v1_1.json", encoding="utf-8") as f:
    rubrica = json.load(f)
trechos = rubrica["trechos"]

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
vetores = embedder.encode([t["texto"] for t in trechos], convert_to_tensor=True)

def recuperar(consulta, k=3):
    v = embedder.encode(consulta, convert_to_tensor=True)
    scores = util.cos_sim(v, vetores)[0]
    achados = []
    for i in scores.argsort(descending=True).tolist():
        if trechos[i]["categoria"] == "geral":
            continue
        achados.append(trechos[i])
        if len(achados) == k:
            break
    return achados

print(f"rubrica {rubrica['versao']}: {len(trechos)} trechos indexados ✓")

In [ ]:
# Etapa 4 — Modelo (Qwen3-VL-8B em 4-bit, como nos notebooks anteriores)
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODELO = "Qwen/Qwen3-VL-8B-Instruct"
quantizacao = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = Qwen3VLForConditionalGeneration.from_pretrained(
    MODELO, quantization_config=quantizacao, device_map="auto"
)
processador = AutoProcessor.from_pretrained(MODELO)

def gerar(conteudo, max_tokens=400):
    conversa = [{"role": "user", "content": conteudo}]
    entradas = processador.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return processador.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extrair_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

print("modelo carregado ✓")

## Etapa 5 — Observação visual dos 20 (~15 min)

Mesmo prompt v2 dos notebooks anteriores (estável desde a E5). A célula imprime o progresso a cada objeto.

In [ ]:
PROMPT_OBSERVACAO_V2 = (
    "Descreva APENAS o que está visível nesta fotografia de um objeto de museu:\n"
    "- formas, cores e materiais aparentes — incluindo cores de bordas, faixas e acabamentos, não só as dominantes;\n"
    "- a posição/orientação do objeto (de pé, inclinado, deitado) e se partes internas (boca, interior, verso) estão visíveis;\n"
    "- o enquadramento: o objeto aparece inteiro ou só um detalhe/close?;\n"
    "- o fundo e qualquer artefato de estúdio (etiqueta, numeração, cartela de cores, régua, suporte).\n"
    "NÃO invente o que não dá para ver. Se algo estiver ilegível ou incerto, diga isso em vez de estimar. "
    "Responda em português."
)

for n, obj in enumerate(objetos, 1):
    obj["observacao"] = gerar(
        [{"type": "image", "image": obj["foto"]}, {"type": "text", "text": PROMPT_OBSERVACAO_V2}]
    )
    print(f"[{n}/{len(objetos)}] {obj['titulo']} observado ✓")

## Etapa 6 — Redação estruturada v6: dois bugs de prompt fechados (~20 min)

A redação produz as **três saídas do sistema de uma vez** (alt-text + descrição do objeto + flags, num único JSON por objeto). O prompt v6 mantém tudo que a v5 acertou e fecha os 2 bugs achados na análise da v2:

- **artefato nunca aparece**: agora a regra vale explicitamente para as DUAS saídas (alt-text e nível 2), cada uma com seu próprio exemplo errado×certo — antes só o alt-text tinha a regra escrita;
- **atribuição ao registro**: virou exigência explícita — o 1º parágrafo do nível 2 DEVE conter uma marca de atribuição ("segundo o registro do museu", "o registro informa que", "de acordo com o catálogo"...) antes do primeiro fato do catálogo. Antes, isso só era sugerido via um exemplo específico (o Pote), e o modelo generalizou mal para os outros 19 objetos.

O modelo continua sem ver a imagem nesta etapa — recebe a observação, o registro completo e as diretrizes recuperadas da rubrica v1.1.

In [ ]:
PROMPT_REDACAO_V6 = """Você escreve descrições de acessibilidade para o acervo digital de um museu, lidas por pessoas cegas via leitor de tela. Escreva em linguagem cotidiana — NUNCA jargão de catálogo ('globular' → 'arredondado').

OBSERVAÇÃO VISUAL DA FOTOGRAFIA (única fonte do que é visível):
{observacao}

REGISTRO DO MUSEU (fatos do catálogo — o título nomeia o objeto):
{registro}

DIRETRIZES PARA ESTE TIPO DE OBJETO (recuperadas da base do projeto):
{diretrizes}

PRODUZA TRÊS SAÍDAS:

A) alt_text — uma frase, máx. 30 palavras, descrevendo a FOTOGRAFIA. Começa pelo objeto (nomeado pelo TÍTULO do registro — nunca rebatize pela aparência) e pelo povo. Enquadramento: objeto INTEIRO → nunca use 'close'; DETALHE → comece com 'Detalhe de...'. Cores: nomeie onde a cor informa (penas, miçangas, pinturas, padronagens — proibido 'colorido', 'tons variados'); em material natural sem tingimento (madeira, palha, cerâmica, cipó), nomeie o MATERIAL em vez da cor. Padrões têm FORMA além de cor (faixas, xadrez, losangos). Se a matéria-prima do registro nomeia as aves das penas, cite as espécies em conjunto ('penas de arara, jaburu e jacu') no lugar da lista de cores. Preserve incerteza (palpite vira termo genérico). ARTEFATO DE ESTÚDIO/INVENTÁRIO NUNCA APARECE (a mesma regra vale para a saída B) — ERRADO: '...com marcação numérica na base.' CERTO: terminar a frase sem citar a marcação (ela vira flag).

B) descricao_objeto — descreve o OBJETO, não a fotografia: PROIBIDO mencionar posição, inclinação, fundo, enquadramento ou a foto. PROIBIDO citar artefato de estúdio/inventário, pela mesma regra da saída A — ERRADO: 'A base apresenta uma marcação numérica, que não é parte do objeto original.' CERTO: a frase termina sem citar a marcação (ela já virou flag). Dois parágrafos:
   1º: abre direto com o objeto e sua função. O parágrafo DEVE conter uma marca explícita de atribuição antes do primeiro fato que não é visível na foto (função, técnica, origem, ano) — use 'segundo o registro do museu', 'o registro informa que', 'de acordo com o catálogo' ou equivalente. Sem essa marca, quem lê não consegue separar o que a foto mostra do que o museu registrou — por isso ela é obrigatória uma vez, mesmo que soe repetitiva. Ex.: 'Um pote cerâmico Karajá que, segundo o registro do museu, era usado no preparo e serviço de alimentos.' PROIBIDO abrir com 'O objeto é...', 'Trata-se de...' ou anunciar 'A função é...'. Depois da atribuição, a aparência: formas, materiais e padrões em palavras comuns (glossário das diretrizes ajuda a traduzir).
   2º: os demais fatos do catálogo (técnica, origem da pena ou do material, ano) em frases naturais ('adquirido em 1977' — nunca 'foi aquisição em'), sem repetir a marca de atribuição do 1º parágrafo e sem repetir o material já dito. Medidas: só uma noção de escala ('peça pequena, cerca de 6 cm de altura') — nunca as três dimensões. Se o registro nomeia as aves das penas, detalhe ave a ave. Significado cultural: só se estiver no registro.
   NUNCA afirme ausências ('sem etiquetas', 'não há sinais de...') — o que não existe simplesmente não aparece no texto.

C) flags — lista do que precisa de revisão humana:
   - tipo 'artefato_estudio': TODO artefato que a observação notou (etiqueta, numeração, cartela, régua, suporte, borda) DEVE virar uma flag — se a observação viu, a flag existe;
   - tipo 'divergencia_imagem_catalogo': algo claramente visível que o registro não menciona, ou objeto visto diferente do que o título nomeia (nesse caso, use o título no texto e registre a divergência aqui);
   - tipo 'metadado_suspeito': valor do registro que parece improvável (dimensão absurda, data impossível).
   Lista vazia [] se não houver nada.

Responda APENAS com JSON: {{"alt_text": "...", "descricao_objeto": "...", "flags": [{{"tipo": "...", "detalhe": "..."}}]}}"""

for n, obj in enumerate(objetos, 1):
    registro_txt = "\n".join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    consulta = f"{obj['titulo']} ({obj['registro']['Categoria']}). {obj['observacao'][:250]}"
    achados = recuperar(consulta)
    obj["diretrizes_usadas"] = [t["id"] for t in achados]
    prompt = PROMPT_REDACAO_V6.format(
        observacao=obj["observacao"],
        registro=registro_txt,
        diretrizes="\n".join(f"- {t['texto']}" for t in achados),
    )
    resposta = gerar([{"type": "text", "text": prompt}], max_tokens=700)
    try:
        saida = extrair_json(resposta)
        obj["alt_text"] = saida["alt_text"]
        obj["descricao_objeto"] = saida["descricao_objeto"]
        obj["flags"] = saida.get("flags", [])
        obj["json_valido"] = True
    except Exception:
        obj["alt_text"], obj["descricao_objeto"], obj["flags"] = resposta, "", []
        obj["json_valido"] = False
    print(f"[{n}/{len(objetos)}] {obj['titulo']}: {obj['alt_text'][:80]}... | flags: {len(obj['flags'])}")

## Etapa 7 — Verificação automática (2 falsos positivos corrigidos)

A análise da v2 achou 2 bugs na própria checagem, não no texto gerado: "o objeto é" estava sendo procurado em QUALQUER posição do texto (pegando frases legítimas como "...e o objeto é pequeno..."), quando a regra só proíbe abrir o texto assim; e "sobre fundo X" estava marcado como foto vazando, quando é vocabulário legítimo de padronagem ("padrões geométricos sobre fundo bege" descreve a peça, não a fotografia). Os dois foram corrigidos: a checagem de abertura agora olha só o início do texto, e os termos de fundo saíram da lista de "foto no nível 2".

As checagens continuam: JSON válido; povo no alt-text; artefato no alt; ≤30 palavras; atribuição ao registro no nível 2 (aceitando as formulações variadas); artefato no nível 2 (novo, fecha o bug 1 do prompt v6); afirmações de ausência; foto vazando para o nível 2; caso-referência do Abano.

In [ ]:
TERMOS_ARTEFATO = ["cartela", "paleta", "numeração", "marcação", "etiqueta", "régua", "suporte"]
ABERTURAS_ETIQUETA = ["o objeto é", "trata-se de"]  # só conta se ABRE o texto, não em qualquer posição
TERMOS_AUSENCIA = ["não há", "sem etiqueta", "sem sinais", "sem evidência", "sem artefatos", "sem marcas"]
TERMOS_FOTO = ["posicionad", "inclinad", "enquadr", "fotografia", "na imagem", "da imagem"]

def tem_atribuicao(texto):
    t = texto.lower()
    return "registro" in t or "catálogo" in t or "catalogo" in t

problemas_totais = 0
for obj in objetos:
    p = []
    if not obj["json_valido"]:
        p.append("JSON inválido")
    povo = obj["registro"]["Povo"]
    if povo and povo.split()[0].lower() not in obj["alt_text"].lower():
        p.append(f"povo '{povo}' ausente do alt")
    for termo in TERMOS_ARTEFATO:
        if termo in obj["alt_text"].lower():
            p.append(f"artefato no alt ('{termo}')")
    if len(obj["alt_text"].split()) > 30:
        p.append(f"{len(obj['alt_text'].split())} palavras")
    d = obj["descricao_objeto"].lower().strip()
    if obj["descricao_objeto"]:
        if not tem_atribuicao(obj["descricao_objeto"]):
            p.append("nível 2 sem atribuição ao registro")
        if any(d.startswith(a) for a in ABERTURAS_ETIQUETA):
            p.append("nível 2 abre com frase-etiqueta")
        if "a função é" in d:
            p.append("frase-etiqueta ('a função é')")
        for termo in TERMOS_ARTEFATO:
            if termo in d:
                p.append(f"artefato no nível 2 ('{termo}')")
        for termo in TERMOS_AUSENCIA:
            if termo in d:
                p.append(f"afirmação de ausência ('{termo}')")
        for termo in TERMOS_FOTO:
            if termo in d:
                p.append(f"foto no nível 2 ('{termo}')")
    obj["problemas"] = p
    problemas_totais += len(p)
    status = "✓" if not p else "⚠ " + "; ".join(p)
    print(f"{obj['id']} {obj['titulo'][:30]:30} {status}")

abano = next(o for o in objetos if o["id"] == 63283)
abano_ok = abano["alt_text"].strip().lower().startswith("detalhe") and tem_atribuicao(abano["descricao_objeto"])
print(f"\nCaso-referência Abano: {'✓ passou' if abano_ok else '✗ FALHOU'}")
print(f"Total: {sum(1 for o in objetos if not o['problemas'])}/{len(objetos)} objetos sem problemas | {sum(len(o['flags']) for o in objetos)} flags geradas")

In [ ]:
# Etapa 8 — Salvar no Drive (arquivo v3 — os resultados da v1 e v2 ficam preservados)
resultado = {
    "notebook": "04_pipeline_completo_v3",
    "modelo": MODELO,
    "embedding": "Qwen/Qwen3-Embedding-0.6B",
    "rubrica_versao": rubrica["versao"],
    "prompt_observacao": PROMPT_OBSERVACAO_V2,
    "prompt_redacao_v6": PROMPT_REDACAO_V6,
    "itens": [
        {"id": o["id"], "titulo": o["titulo"], "registro": o["registro"],
         "observacao": o["observacao"], "alt_text": o["alt_text"],
         "descricao_objeto": o["descricao_objeto"], "flags": o["flags"],
         "diretrizes_usadas": o["diretrizes_usadas"],
         "json_valido": o["json_valido"], "problemas": o["problemas"]}
        for o in objetos
    ],
}
destino = f"{PROJETO}/resultados/04_pipeline_completo_v3.json"
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive ✓  {destino}")

---

## Fim — o que fazer agora

Avise o Claude que o Notebook 04 **v3** terminou — ele busca o resultado no Drive, compara com os lotes anteriores e monta a página de revisão. Se os 2 bugs realmente fecharam, esta é a rodada em que a sua revisão vira só fidelidade visual (o texto mente algo sobre a foto?) — o estilo já deve estar coberto por regra e verificação automática.

**O que este notebook prova:** o ciclo completo do método, incluindo a parte que nem sempre aparece — encontrar e corrigir bugs no próprio prompt e na própria verificação, não só nas respostas do modelo. **O que ainda não prova:** as métricas oficiais nos 40 casos (E8) e o julgamento humano (E10).